# Linear Regression with FHE

This notebook demonstrates linear regression on encrypted data,
covering advanced topics like regularization and model comparison.

## Topics Covered
1. Basic linear regression on encrypted data
2. Comparison with sklearn
3. Feature importance analysis
4. Handling different data scales

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_regression, fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.linear_model import LinearRegression as SklearnLR

import sys
sys.path.insert(0, '..')

from sdk.models import LinearRegression, ModelConfig

## 1. Basic Linear Regression

Let's start with a simple synthetic dataset.

In [ ]:
# Generate data with known coefficients
np.random.seed(42)
n_samples = 500
n_features = 5

# True coefficients
true_weights = np.array([1.5, -2.0, 0.5, 3.0, -1.0])
true_bias = 2.0

# Generate features
X = np.random.randn(n_samples, n_features)

# Generate target with noise
y = X @ true_weights + true_bias + np.random.randn(n_samples) * 0.5

# Normalize
scaler_X = StandardScaler()
scaler_y = StandardScaler()
X_scaled = scaler_X.fit_transform(X)
y_scaled = scaler_y.fit_transform(y.reshape(-1, 1)).flatten()

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_scaled, test_size=0.2, random_state=42
)

print(f"Dataset: {n_samples} samples, {n_features} features")
print(f"True weights: {true_weights}")
print(f"True bias: {true_bias}")

In [ ]:
# Train FHE-compatible model
config = ModelConfig(
    learning_rate=0.1,
    n_epochs=200,
    verbose=False,
)

fhe_model = LinearRegression(config=config)
fhe_model._fit_plaintext(X_train, y_train)

# Predictions
y_pred_fhe = fhe_model._predict_plaintext(X_test)

print("FHE Linear Regression:")
print(f"  Learned weights: {fhe_model.weights.round(3)}")
print(f"  Learned bias: {fhe_model.bias:.3f}")
print(f"  R² Score: {r2_score(y_test, y_pred_fhe):.4f}")
print(f"  MSE: {mean_squared_error(y_test, y_pred_fhe):.4f}")

## 2. Comparison with Scikit-learn

In [ ]:
# Train sklearn model
sklearn_model = SklearnLR()
sklearn_model.fit(X_train, y_train)
y_pred_sklearn = sklearn_model.predict(X_test)

print("Sklearn Linear Regression:")
print(f"  Weights: {sklearn_model.coef_.round(3)}")
print(f"  Bias: {sklearn_model.intercept_:.3f}")
print(f"  R² Score: {r2_score(y_test, y_pred_sklearn):.4f}")
print(f"  MSE: {mean_squared_error(y_test, y_pred_sklearn):.4f}")

In [ ]:
# Compare weights
print("\nWeight Comparison (FHE vs Sklearn):")
print(f"{'Feature':<10} {'FHE':>10} {'Sklearn':>10} {'Diff':>10}")
print("-" * 42)
for i in range(n_features):
    diff = abs(fhe_model.weights[i] - sklearn_model.coef_[i])
    print(f"Feature {i:<3} {fhe_model.weights[i]:>10.4f} {sklearn_model.coef_[i]:>10.4f} {diff:>10.4f}")

bias_diff = abs(fhe_model.bias - sklearn_model.intercept_)
print(f"{'Bias':<10} {fhe_model.bias:>10.4f} {sklearn_model.intercept_:>10.4f} {bias_diff:>10.4f}")

## 3. Training Convergence Analysis

In [ ]:
# Train with different learning rates
learning_rates = [0.01, 0.05, 0.1, 0.2]
histories = {}

for lr in learning_rates:
    model = LinearRegression(config=ModelConfig(learning_rate=lr, n_epochs=100))
    model._fit_plaintext(X_train, y_train)
    histories[lr] = model.history.losses

# Plot convergence
plt.figure(figsize=(10, 5))
for lr, losses in histories.items():
    plt.plot(losses, label=f'LR={lr}')
plt.xlabel('Epoch')
plt.ylabel('Loss (MSE)')
plt.title('Training Convergence by Learning Rate')
plt.legend()
plt.yscale('log')
plt.grid(True, alpha=0.3)
plt.show()

## 4. Real Dataset: California Housing

In [ ]:
# Load California Housing dataset
housing = fetch_california_housing()
X_housing = housing.data[:2000]  # Use subset for speed
y_housing = housing.target[:2000]

print(f"California Housing Dataset")
print(f"  Samples: {len(X_housing)}")
print(f"  Features: {X_housing.shape[1]}")
print(f"  Feature names: {housing.feature_names}")

In [ ]:
# Prepare data
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_h = scaler_X.fit_transform(X_housing)
y_h = scaler_y.fit_transform(y_housing.reshape(-1, 1)).flatten()

X_train_h, X_test_h, y_train_h, y_test_h = train_test_split(
    X_h, y_h, test_size=0.2, random_state=42
)

# Train
housing_model = LinearRegression(config=ModelConfig(learning_rate=0.1, n_epochs=200))
housing_model._fit_plaintext(X_train_h, y_train_h)

# Evaluate
y_pred_h = housing_model._predict_plaintext(X_test_h)

print(f"\nResults on California Housing:")
print(f"  R² Score: {r2_score(y_test_h, y_pred_h):.4f}")
print(f"  MAE: {mean_absolute_error(y_test_h, y_pred_h):.4f}")

In [ ]:
# Feature importance
importance = np.abs(housing_model.weights)
indices = np.argsort(importance)[::-1]

print("\nFeature Importance (by absolute weight):")
for i, idx in enumerate(indices):
    print(f"  {i+1}. {housing.feature_names[idx]}: {importance[idx]:.4f}")

# Plot
plt.figure(figsize=(10, 5))
plt.bar(range(len(importance)), importance[indices])
plt.xticks(range(len(importance)), [housing.feature_names[i] for i in indices], rotation=45)
plt.ylabel('Absolute Weight')
plt.title('Feature Importance')
plt.tight_layout()
plt.show()

## 5. Early Stopping

In [ ]:
# Train with early stopping
config_early = ModelConfig(
    learning_rate=0.1,
    n_epochs=500,
    early_stopping_patience=10,
    tolerance=1e-6,
)

model_early = LinearRegression(config=config_early)
model_early._fit_plaintext(X_train, y_train)

print(f"Early Stopping:")
print(f"  Stopped at epoch: {model_early.history.epochs}")
print(f"  Final loss: {model_early.history.losses[-1]:.6f}")
print(f"  R² Score: {r2_score(y_test, model_early._predict_plaintext(X_test)):.4f}")

## Summary

Key takeaways:
- FHE Linear Regression achieves comparable results to sklearn
- Learning rate affects convergence speed
- Data normalization is critical for FHE operations
- Early stopping prevents overfitting and saves computation